# Warm-up 3A · Vietnamese Forest Bird Calls · Nhận diện tiếng chim rừng Việt Nam (Audio · Âm thanh, multi-label)

| | 🇬🇧 English | 🇻🇳 Tiếng Việt |
|---|---|---|
| **Task** | For every 4-second recording, predict the probability that each of the 10 bird species below is calling (0, 1, 2 or 3 species can be present). | Với mỗi đoạn ghi âm 4 giây, dự đoán xác suất mỗi loài trong 10 loài chim dưới đây có mặt (có thể có 0, 1, 2 hoặc 3 loài). |
| **Data** | `train/` 1,300 *focal* recordings (a birder points the microphone at one bird), labelled with `primary_label` and, when noted, `secondary_labels`. `public_test/` 400 + `private_test/` 400 *soundscapes* from passive recorders left in Cúc Phương and Cát Tiên national parks (no labels). Mono WAV, 8 kHz, 16-bit. | `train/` 1.300 bản ghi *focal* (người ghi âm hướng micro vào một con chim), có nhãn `primary_label` và `secondary_labels` (nếu được ghi chú). `public_test/` 400 + `private_test/` 400 bản ghi *soundscape* từ máy ghi âm tự động đặt ở VQG Cúc Phương và Cát Tiên (không có nhãn). WAV mono, 8 kHz, 16-bit. |
| **Metric** | Macro-averaged ROC-AUC over the species that occur at least once in the split, **higher is better**. Ranking uses the **private** score. | ROC-AUC trung bình (macro) trên các loài xuất hiện ít nhất một lần trong tập, **càng cao càng tốt**. Xếp hạng theo điểm **private**. |
| **Submission** | `public_submission.csv`, `private_submission.csv` with columns `id` + one probability column per species, in the order of `SPECIES` | `public_submission.csv`, `private_submission.csv` gồm cột `id` + một cột xác suất cho mỗi loài, theo đúng thứ tự `SPECIES` |
| **Rules** | PyTorch model, no external audio data (torchvision pretrained weights are allowed). | Mô hình PyTorch, không dùng dữ liệu âm thanh bên ngoài (được dùng trọng số pretrained của torchvision). |

$$\text{Score}=\frac{1}{|C^{+}|}\sum_{c\in C^{+}}\text{AUC}_c,\qquad C^{+}=\{c:\ \text{species } c \text{ is present in at least one clip of the split}\}$$

| Code · Mã | English | Tiếng Việt | Scientific name |
|---|---|---|---|
| `blnori1` | Black-naped oriole | Vàng anh | *Oriolus chinensis* |
| `colsco1` | Collared scops owl | Cú mèo khoang cổ | *Otus lettia* |
| `grecou1` | Greater coucal | Bìm bịp lớn | *Centropus sinensis* |
| `grehor1` | Great hornbill | Hồng hoàng | *Buceros bicornis* |
| `greyel` | Greater yellownape (woodpecker) | Gõ kiến xanh gáy vàng | *Chrysophlegma flavinucha* |
| `orimag1` | Oriental magpie-robin | Chích chòe than | *Copsychus saularis* |
| `rewbul` | Red-whiskered bulbul | Chào mào | *Pycnonotus jocosus* |
| `silphe` | Silver pheasant | Gà lôi trắng | *Lophura nycthemera* |
| `spodov` | Spotted dove | Cu gáy | *Spilopelia chinensis* |
| `wcrlau1` | White-crested laughingthrush | Khướu đầu trắng | *Garrulax leucolophus* |

**Columns · Các cột:** `train.csv`: `id, filename, primary_label, secondary_labels, site, hour` (`secondary_labels`: space-separated, often empty · cách nhau bởi dấu cách, thường để trống).
`public_test.csv` / `private_test.csv`: `id, filename, site, hour`. `filename` is relative to the split folder · `filename` là đường dẫn tương đối trong thư mục của tập.

Inspired by / Lấy cảm hứng từ: [BirdCLEF+ 2025 (Kaggle · LifeCLEF 2025)](https://www.kaggle.com/competitions/birdclef-2025) and BirdCLEF 2024. All audio here is synthetic · Toàn bộ âm thanh ở đây là dữ liệu tổng hợp.

🇬🇧 This baseline runs end-to-end and submits a result. Beating it is up to you.
🇻🇳 Baseline này chạy trọn vẹn và nộp được kết quả. Vượt qua nó là việc của bạn.

## 1. Setup and Imports · Cài đặt và thư viện

In [ ]:
import os
import re
import shutil
import unicodedata
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import torchaudio
from scipy.io import wavfile
from sklearn.metrics import roc_auc_score

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration and Hyperparameters · Cấu hình và siêu tham số
🇬🇧 Nothing to fill in here: the data downloads itself, and your team comes from your account on the site when you submit (section 7).
🇻🇳 Không cần điền gì ở đây: dữ liệu tự tải về, và đội của bạn được lấy từ tài khoản trên trang web khi nộp bài (phần 7).


In [ ]:
# ---- Data / Dữ liệu ----
# The dataset downloads itself from GitHub straight onto this Colab session's own
# disk. No Google Drive and no mounting: reading thousands of small files over a
# Drive mount is what made the old flow slow, and local disk is several times
# faster per epoch.
# Bộ dữ liệu tự tải từ GitHub vào ổ đĩa của phiên Colab này. Không cần Drive.
TASK    = "3A"
FOLDER  = "3A_bird_audio"
RELEASE = "https://github.com/ftulabs/olympiad-datasets/releases/download/warmup-3"

# Already have the folder? Point this at the parent of 3A_bird_audio/ and nothing is
# downloaded. / Đã có sẵn thư mục? Trỏ vào thư mục cha của 3A_bird_audio/.
ROOT_OVERRIDE = ""

import os, subprocess, time

ROOT = ROOT_OVERRIDE or "/content/olympiad"
if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    os.makedirs(ROOT, exist_ok=True)
    zip_path = f"{ROOT}/{FOLDER}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {FOLDER}.zip / Đang tải ...", flush=True)
        t0 = time.time()
        subprocess.run(["curl", "-fsSL", "--retry", "3", "-o", zip_path,
                        f"{RELEASE}/{FOLDER}.zip"], check=True)
        print(f"  {os.path.getsize(zip_path)/1e6:.0f} MB in {time.time()-t0:.0f}s", flush=True)
    print("Extracting / Đang giải nén ...", flush=True)
    subprocess.run(["unzip", "-nq", zip_path, "-d", ROOT], check=True)

if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    raise FileNotFoundError(
        f"{ROOT}/{FOLDER}/dataset is missing after the download. Re-run this cell. "
        f"/ Thiếu dữ liệu sau khi tải. Hãy chạy lại ô này."
    )
print(f"Data root / Thư mục dữ liệu: {ROOT}")

DATA_DIR = f"{ROOT}/3A_bird_audio/dataset"
TRAIN_DIR = f"{DATA_DIR}/train"
PUBLIC_DIR = f"{DATA_DIR}/public_test"
PRIVATE_DIR = f"{DATA_DIR}/private_test"
RESULTS_DIR = f"{ROOT}/results"  # shared submission folder / thư mục nộp bài chung
SAVE_DIR = "./checkpoints"

# ---- Hyperparameters / Siêu tham số ----
SAMPLE_RATE = 8000
N_MELS = 64
BATCH_SIZE = 32
EPOCHS = 15
LR = 1e-3
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

os.makedirs(SAVE_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3. Data Loading and Log-Mel Spectrograms · Đọc dữ liệu và phổ log-mel

In [ ]:
train_df = pd.read_csv(f"{TRAIN_DIR}/train.csv")
public_df = pd.read_csv(f"{PUBLIC_DIR}/public_test.csv")
private_df = pd.read_csv(f"{PRIVATE_DIR}/private_test.csv")
print(f"Train: {len(train_df)} | Public test: {len(public_df)} | Private test: {len(private_df)}")

# Submission column order (do not change) / Thứ tự cột khi nộp bài (không được đổi)
SPECIES = ["blnori1", "colsco1", "grecou1", "grehor1", "greyel", "orimag1",
           "rewbul", "silphe", "spodov", "wcrlau1"]
print("SPECIES =", SPECIES)


def load_wave(path):
    """Read an int16 WAV file as float32 in [-1, 1]. / Đọc file WAV int16 thành float32 trong [-1, 1]."""
    sr, x = wavfile.read(path)
    assert sr == SAMPLE_RATE
    return torch.tensor(x.astype(np.float32) / 32768.0)


# Log-mel spectrogram: (N, 1, N_MELS, frames) / Phổ log-mel
mel = torchaudio.transforms.MelSpectrogram(sample_rate=SAMPLE_RATE, n_fft=512, hop_length=128, n_mels=N_MELS).to(DEVICE)
to_db = torchaudio.transforms.AmplitudeToDB().to(DEVICE)


@torch.no_grad()
def load_mels(df, split_dir, batch=64):
    out = []
    for i in range(0, len(df), batch):
        waves = torch.stack([load_wave(f"{split_dir}/{f}") for f in df["filename"].iloc[i:i + batch]])
        out.append(to_db(mel(waves.to(DEVICE))).cpu())
    return torch.cat(out).unsqueeze(1)


X_all = load_mels(train_df, TRAIN_DIR)
X_public = load_mels(public_df, PUBLIC_DIR)
X_private = load_mels(private_df, PRIVATE_DIR)

# Targets: one-hot of the primary label / Nhãn: one-hot theo loài chính (primary_label)
Y_all = torch.zeros(len(train_df), len(SPECIES))
Y_all[torch.arange(len(train_df)), torch.tensor(train_df["primary_label"].map(SPECIES.index).values)] = 1.0

# Random 80/20 train/validation split / Chia ngẫu nhiên 80/20 train/validation
perm = np.random.RandomState(SEED).permutation(len(train_df))
val_idx, tr_idx = perm[: len(perm) // 5], perm[len(perm) // 5:]

# Standardise with train statistics / Chuẩn hoá theo thống kê của tập train
mean, std = X_all[tr_idx].mean(), X_all[tr_idx].std()
X_all, X_public, X_private = [(x - mean) / std for x in (X_all, X_public, X_private)]

X_train, y_train = X_all[tr_idx], Y_all[tr_idx]
X_val, y_val = X_all[val_idx], Y_all[val_idx]
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
print(f"Train: {tuple(X_train.shape)} | Validation: {tuple(X_val.shape)} | Public: {tuple(X_public.shape)}")
train_df.head()

## 4. Model Architecture · Kiến trúc mô hình

In [ ]:
def conv_block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=3, padding=1),
        nn.BatchNorm2d(c_out),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )


class BirdCNN(nn.Module):
    """Small CNN on log-mel spectrograms. / CNN nhỏ trên phổ log-mel."""

    def __init__(self, n_classes=len(SPECIES)):
        super().__init__()
        self.features = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, 64), conv_block(64, 128))
        self.head = nn.Linear(128, n_classes)

    def forward(self, x):
        x = self.features(x).mean(dim=(2, 3))  # global average pooling / gộp trung bình toàn cục
        return self.head(x)  # logits


model = BirdCNN().to(DEVICE)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Training · Huấn luyện

In [ ]:
@torch.no_grad()
def predict(model, X, batch=128):
    """Sigmoid probabilities, shape (N, n_species). / Xác suất sigmoid, kích thước (N, số loài)."""
    model.eval()
    return torch.cat([torch.sigmoid(model(X[i:i + batch].to(DEVICE))).cpu() for i in range(0, len(X), batch)]).numpy()


def macro_auc(y_true, y_prob):
    """Macro ROC-AUC over species with at least one positive. / ROC-AUC trung bình trên các loài có mẫu dương."""
    y_true = np.asarray(y_true)
    cols = [c for c in range(y_true.shape[1]) if 0 < y_true[:, c].sum() < len(y_true)]
    return float(np.mean([roc_auc_score(y_true[:, c], y_prob[:, c]) for c in cols]))


criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_auc = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)

    val_auc = macro_auc(y_val, predict(model, X_val))
    msg = f"Epoch {epoch:02d} | Loss={total_loss / len(X_train):.4f} | Val macro AUC={val_auc:.4f}"
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pt"))
        msg += "  ✓ best model saved"
    print(msg)

print(f"\nBest validation macro AUC: {best_auc:.4f}")

## 6. Predict Public & Private Test · Dự đoán tập Public & Private
🇬🇧 Creates `public_submission.csv` / `private_submission.csv` (+ `.zip`), like the official competition.
🇻🇳 Tạo `public_submission.csv` / `private_submission.csv` (+ `.zip`), giống định dạng cuộc thi chính thức.

In [ ]:
def export(ids, probs, csv_path):
    """Save <id, one column per species> CSV and zip it. / Lưu CSV <id, mỗi loài một cột> và nén zip."""
    sub = pd.DataFrame(probs, columns=SPECIES)
    sub.insert(0, "id", ids.values)
    sub.to_csv(csv_path, index=False)
    zip_path = csv_path.replace(".csv", ".zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname=os.path.basename(csv_path))
    print(f"Saved {csv_path} and {zip_path}")


model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "best_model.pt"), map_location=DEVICE))
export(public_df["id"], predict(model, X_public), "public_submission.csv")
export(private_df["id"], predict(model, X_private), "private_submission.csv")
pd.read_csv("public_submission.csv").head()

## 7. Submit · Nộp bài

🇬🇧 Your team comes from your account on [seleccion.ftds.online](https://seleccion.ftds.online/), not from a name typed here. The cell below prints a short code once per Colab session; type it on the site and this notebook submits as you.

🇻🇳 Đội của bạn lấy từ tài khoản trên [seleccion.ftds.online](https://seleccion.ftds.online/). Ô bên dưới in ra một mã ngắn mỗi phiên Colab; nhập mã đó trên trang web.


In [ ]:
def check_submission(csv_path, test_df):
    """Validate the file format before submitting. / Kiểm tra định dạng file trước khi nộp."""
    sub = pd.read_csv(csv_path)
    assert list(sub.columns) == ["id", *SPECIES], f"{csv_path}: columns must be ['id', *SPECIES] in this order"
    assert len(sub) == len(test_df) and set(sub["id"]) == set(test_df["id"]), f"{csv_path}: ids do not match the test set"
    assert sub[SPECIES].notna().all().all(), f"{csv_path}: contains missing values / có giá trị bị thiếu"
    assert np.isfinite(sub[SPECIES].to_numpy(dtype=float)).all(), f"{csv_path}: values must be finite / giá trị phải hữu hạn"


check_submission("public_submission.csv", public_df)
check_submission("private_submission.csv", private_df)


In [ ]:
# ===================================================================
#  Olympiad AI - submit.  Paste at the END of your notebook, then run.
#  No key to copy: the first run prints a code you enter on the site.
# ===================================================================
BASE    = "https://seleccion.ftds.online"
PROBLEM = "3A"          # which problem this notebook is for
CONTEST = ""            # blank = the contest you are entered in
PUBLIC  = "public_submission.csv"
PRIVATE = "private_submission.csv"
# -------------------------------------------------------------------
import os, time, requests

token = globals().get("_OLY_TOKEN")
if not token:
    start = requests.post(f"{BASE}/api/pair/start", timeout=30).json()
    print(f"\n  1. open   {start['enter_at']}")
    print(f"  2. enter  {start['code']}\n  waiting...", flush=True)
    for _ in range(int(start["expires_in"]) // 3):
        time.sleep(3)
        got = requests.get(f"{BASE}/api/pair/poll",
                           params={"code": start["code"]}, timeout=30).json()
        if got.get("ready"):
            token = globals()["_OLY_TOKEN"] = got["token"]
            print("  paired ✓ (this notebook stays paired for the session)")
            break
    else:
        raise SystemExit("pairing timed out - run this cell again")

missing = [f for f in (PUBLIC, PRIVATE) if not os.path.exists(f)]
if PUBLIC in missing:
    raise SystemExit(f"{PUBLIC} not found - run the cell that writes it first")

files = {"public": open(PUBLIC, "rb")}
if PRIVATE not in missing:
    files["private"] = open(PRIVATE, "rb")
try:
    r = requests.post(f"{BASE}/api/submit", headers={"X-Oly-Token": token},
                      data={"problem": PROBLEM, "contest": CONTEST},
                      files=files, timeout=600)
finally:
    for f in files.values():
        f.close()

try:
    d = r.json()
except ValueError:
    raise SystemExit(f"server {r.status_code}: {r.text[:300]}")

if r.status_code == 429:
    print(f"⏳ cooldown - wait {d.get('retry_after')}s and run again")
elif r.status_code == 401:
    globals().pop("_OLY_TOKEN", None)
    print(f"❌ {d.get('detail')}  (run this cell again to re-pair)")
elif r.status_code != 200:
    print(f"❌ {r.status_code}: {d.get('detail') or d.get('message')}")
elif d["public"]["status"] != "ok":
    # the grader's own message about your file - it says what to fix
    print(f"❌ {d['public']['status'].upper()}: {d['public']['message']}")
else:
    p, c = d["public"], (d.get("counting") or {})
    print(f"✅ {d['problem']} · {d['title']}")
    print(f"   {d['metric']} = {p['score']:.4f}   {p['tier'] or '(below the lowest tier)'}")
    print(f"   entry #{d['entries']}   ·   rank: {d['rank']}")
    if c.get("score") is not None and abs(c["score"] - p["score"]) > 1e-12:
        print(f"   counting entry: {c['score']:.4f} {c['tier']}")
    print(f"   {BASE}/")
